# פרק 7: בניית יישומי צ'אט
## התחלה מהירה עם Microsoft Foundry Models API

פנקס זה מותאם מ-[מאגר הדוגמאות של Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) שכולל פנקסים לגישה לשירותי [Azure OpenAI](notebook-azure-openai.ipynb).

> **הערה:** GitHub Models יפסיק לפעול בסוף יולי 2026. פנקס זה מיועד כעת ל-[Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), שמציע את אותו קטלוג דגמים חינמי לנסות וניסיון עם Azure AI Inference SDK.


# סקירה  
"מודלים שפתיים גדולים הם פונקציות שעושות מיפוי מטקסט לטקסט. כאשר מקבלים מחרוזת טקסט קלט, מודל שפה גדול מנסה לנבא את הטקסט שיבוא לאחר מכן"(1). מחברת ה-"התחלה מהירה" הזו תציג למשתמשים מושגים כלליים של LLM, דרישות חבילה מרכזיות להתחלה עם AML, מבוא רך לעיצוב פקודות, וכמה דוגמאות קצרות לשימושים שונים. 


## תוכן העניינים  

[סקירה כללית](#overview)  
[איך להשתמש בשירות OpenAI](#how-to-use-openai-service)  
[1. יצירת שירות OpenAI שלך](#1.-creating-your-openai-service)  
[2. התקנה](#2.-installation)    
[3. אישורים](#3.-credentials)  

[מקרי שימוש](#use-cases)    
[1. סיכום טקסט](#1.-summarize-text)  
[2. סיווג טקסט](#2.-classify-text)  
[3. יצירת שמות מוצרים חדשים](#3.-generate-new-product-names)  
[4. כוונון מדויק של מסווג](#4.fine-tune-a-classifier)  

[מקורות](#references)


### בנה את הפקודה הראשונה שלך  
תרגיל קצר זה יספק מבוא בסיסי להגשת פקודות למודל ב-Microsoft Foundry Models למשימה פשוטה של "סיכום".


**שלבים**:  
1. התקן את הספרייה `azure-ai-inference` בסביבת הפייתון שלך, אם עדיין לא עשית זאת.  
2. טען ספריות עזר סטנדרטיות והגדר את האישורים שלך ל-Microsoft Foundry Models.  
3. בחר מודל למשימה שלך  
4. צור פקודה פשוטה למודל  
5. הגש את הבקשה שלך לממשק ה-API של המודל!


### 1. התקן את `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. ייבוא ספריות עזר ויצירת אישורים


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. מציאת הדגם הנכון  
דגמים כמו GPT-4o ו-GPT-4o מיני יכולים להבין וליצור שפה טבעית, וזמינים בקטלוג דגמי Microsoft Foundry לצד דגמים מ-Meta, Mistral, Cohere ו-Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. עיצוב הפקודה  

"הקסם של מודלי שפה גדולים הוא שבאימונם על הפחתת שגיאת החיזוי הזו על פני כמויות עצומות של טקסט, המודלים למעשה לומדים מושגים שימושיים לחיזויים אלה. לדוגמה, הם לומדים מושגים כמו"(1):

* איך לאיית
* איך דקדוק עובד
* איך לנסח מחדש
* איך לענות על שאלות
* איך לנהל שיחה
* איך לכתוב בשפות רבות
* איך לקודד
* וכו'.

#### איך לשלוט במודל שפה גדול  
"מבין כל הקלטים למודל שפה גדול, גורם הטקסט של הפקודה הוא ללא ספק ההשפעה הגדולה ביותר(1).

ניתן להנחות מודלי שפה גדולים להפיק פלט בכמה דרכים:

הוראה: תגיד למודל מה שאתה רוצה
השלמה: עורר את המודל להשלים את התחלת מה שאתה רוצה
הדגמה: הצג למודל מה אתה רוצה, עם אחד מהבאים:
כמה דוגמאות בהפעלה
מאות או אלפי דוגמאות במערך אימון של כיוון עדין"



#### יש שלוש הנחיות בסיסיות ליצירת פקודות:

**הראה וספר**. היה ברור לגבי מה שאתה רוצה, בין אם באמצעות הוראות, דוגמאות, או שילוב של השניים. אם אתה רוצה שהמודל ידרג רשימת פריטים לפי סדר אלפביתי או יסווג פסקה לפי רגש, הראה לו שזה מה שאתה רוצה.

**ספק נתונים איכותיים**. אם אתה מנסה לבנות מסווג או לגרום למודל לעקוב אחרי דפוס, וודא שיש מספיק דוגמאות. הקפד לקרוא בעיון את הדוגמאות שלך — בדרך כלל המודל חכם מספיק כדי לראות טעויות איות בסיסיות ולתת לך תגובה, אבל ייתכן שהוא גם יחשוב שזה בכוונה וזה יכול להשפיע על התגובה.

**בדוק את ההגדרות שלך.** ההגדרות temperature ו-top_p שולטות עד כמה המודל דטרמיניסטי ביצירת התגובה. אם אתה מבקש תגובה שבה יש תשובה נכונה אחת בלבד, תרצה להוריד הגדרות אלה. אם אתה מחפש תגובות מגוונות יותר, ייתכן שתרצה להעלות אותן. הטעות הנפוצה ביותר שבה עושים אנשים עם ההגדרות האלה היא להניח שהן "שליטה על תחכום" או "יצירתיות".


מקור: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. שלח!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### חזור על אותה קריאה, איך התוצאות מושוות?  


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## לסכם טקסט  
#### אתגר  
סכם טקסט על ידי הוספת 'tl;dr:' לסוף קטע טקסט. שים לב כיצד הדגם מבין לבצע מספר משימות ללא הוראות נוספות. אתה יכול להתנסות עם הנחיות תיאוריות יותר מ-tl;dr כדי לשנות את התנהגות הדגם ולהתאים אישית את הסיכום שתקבל(3).  

עבודה עדכנית הראתה שיפורים משמעותיים במשימות ובמדדים רבים של NLP על ידי אימון מוקדם על מאגר טקסטים גדול ואחריו התאמה מדויקת למשימה ספציפית. בעוד שבדרך כלל הארכיטקטורה אינה תלויה במשימה, שיטה זו עדיין דורשת מערכי נתונים מותאמים למשימה עם אלפי או עשרות אלפי דוגמאות. לעומת זאת, אנשים בדרך כלל יכולים לבצע משימה שפתית חדשה רק מכמה דוגמאות בודדות או מהוראות פשוטות - משהו שמערכות NLP עכשוויות עדיין מתקשות לעשות במידה רבה. כאן אנו מראים כי הגדלת גודל דגמי השפה משפרת משמעותית את הביצועים במשימות כלליות עם מעט דוגמאות, ולפעמים אף מגיעה לתחרות עם שיטות התאמה מדויקת מהמצב האמנותי הקודם.  



תקציר  


# תרגילים למספר מקרים של שימוש  
1. לסכם טקסט  
2. לסווג טקסט  
3. ליצור שמות מוצרים חדשים


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## סיווג טקסט  
#### אתגר  
סווג פריטים לקטגוריות שניתנות בזמן ההסקה. בדוגמה הבאה, אנו מספקים הן את הקטגוריות והן את הטקסט לסיווג בהנחיה (*playground_reference). 

פנייה של לקוח: שלום, אחד המקשים במקלדת הלפטופ שלי נשבר לאחרונה ואצטרך תחליף:

קטגוריה מסווגת:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## הפעל יצירת שמות מוצרים חדשים
#### אתגר
צור שמות למוצרים מתוך מילים לדוגמה. כאן אנו כוללים בפרומפט מידע על המוצר שאנו עומדים ליצור לו שמות. כמו כן, אנו מספקים דוגמה דומה כדי להראות את הדפוס שאנו רוצים לקבל. הגדרנו גם ערך טמפרטורה גבוה כדי להגדיל את המקריות ואת התגובות החדשניות.

תיאור מוצר: מכשיר להכנת מילקשייק ביתי
מילים זרע: מהיר, בריא, קומפקטי.
שמות מוצר: HomeShaker, Fit Shaker, QuickShake, Shake Maker

תיאור מוצר: זוג נעליים שיכולות להתאים לכל מידה של רגל.
מילים זרע: ניתן להתאים, מתאים, אונימיט.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# מקורות  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Best practices for fine-tuning GPT models to classify text](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# לעזרה נוספת  
[צוות המסחור של OpenAI](AzureOpenAITeam@microsoft.com) 


# תורמים
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
